In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

movies = pd.read_csv("ml-latest-small/movies.csv")
ratings = pd.read_csv("ml-latest-small/ratings.csv")

# Prepare movie titles for searching
title_vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5))
title_matrix = title_vectorizer.fit_transform(movies["title"])

# Prepare genres for comparing movies
genre_vectorizer = TfidfVectorizer(token_pattern=r"[^|]+")
genre_matrix = genre_vectorizer.fit_transform(movies["genres"])

# Calculate average ratings
movie_ratings = ratings.groupby("movieId").agg(
    avg_rating=("rating", "mean"),
    total_ratings=("rating", "count")
).reset_index()


def search_movie(name):
    query = title_vectorizer.transform([name])
    scores = cosine_similarity(query, title_matrix).flatten()

    best_match = scores.argmax()

    if scores[best_match] < 0.25:
        print("Movie not found. Try another name.")
        return None

    return movies.iloc[best_match]["movieId"]



def recommend_movies(movie_id):
    movie_index = movies.index[movies["movieId"] == movie_id][0]
    selected_movie = movies.iloc[movie_index]
    selected_title = selected_movie["title"]

    # Find movies with a similar franchise name
    franchise_name = selected_title.rsplit(" (", 1)[0]

    franchise_movies = movies[
        movies["title"].str.contains(
            franchise_name, case=False, regex=False
        )
    ].copy()

    franchise_movies = franchise_movies[
        franchise_movies["movieId"] != movie_id
    ]

    # Compare genres
    scores = cosine_similarity(
        genre_matrix[movie_index],
        genre_matrix
    ).flatten()

    scores[movie_index] = -1

    recommendations = movies.copy()
    recommendations["similarity"] = scores

    recommendations = recommendations.merge(
        movie_ratings, on="movieId", how="left"
    )

    recommendations = recommendations[
        (recommendations["similarity"] > 0) &
        (recommendations["total_ratings"] >= 10)
    ]

    recommendations = recommendations.sort_values(
        ["similarity", "avg_rating"],
        ascending=False
    )

    # Remove franchise movies from the general recommendations
    recommendations = recommendations[
        ~recommendations["movieId"].isin(franchise_movies["movieId"])
    ]

    print("\nSelected Movie:")
    print(selected_title)

    if not franchise_movies.empty:
        print("\nOther Movies in the Same Franchise:\n")
        print(franchise_movies["title"].to_string(index=False))

    print("\nOther Similar Movies:\n")
    print(recommendations[
        ["title", "genres", "similarity", "avg_rating"]
    ].head(10).to_string(index=False))

movie_name = input("Enter a movie name: ")

movie_id = search_movie(movie_name)

if movie_id is not None:
    recommend_movies(movie_id)

